# 0.2 Bandwidth Limit Scenarios

Derive candidate byte-denominated bandwidth limits for bandwidth resource, then map those byte caps into EIP-7999 resource gas limits. Targets are deliberately left for later fee-market tuning.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bandwidth_limits.eip7999_metering import BandwidthMeteringConfig
from bandwidth_limits.propagation import CONSERVATIVE_P90, EMPIRICAL_P90
from bandwidth_limits.scenarios import GLAMSTERDAM_NO_8279, GLAMSTERDAM_PLUS_8279
from bandwidth_limits.sweep import (
    add_propagation_times,
    best_strategy_sweep,
    eip7999_limit_candidates,
    historical_bandwidth_usage,
    safe_payload_cap_sweep,
)
from bandwidth_limits.worst_case import sweep_strategies

## Part 1: Worst-Case Strategy Sweep

In [2]:
GAS_LIMITS = [60_000_000, 100_000_000, 150_000_000, 200_000_000, 300_000_000, 450_000_000]
SCHEDULES = [GLAMSTERDAM_NO_8279, GLAMSTERDAM_PLUS_8279]

all_strategies = sweep_strategies(GAS_LIMITS, SCHEDULES)
best_table = best_strategy_sweep(GAS_LIMITS, SCHEDULES)
best_display_cols = [
    "execution_gas_limit",
    "schedule",
    "best_strategy",
    "calldata_bytes",
    "bal_bytes",
    "tx_access_list_bytes",
    "total_payload_bytes",
    "total_payload_mib",
    "gas_used",
]
display(best_table[best_display_cols])
display(all_strategies[["execution_gas_limit", "schedule", "strategy", "total_payload_bytes", "gas_used", "is_best"]])

,execution_gas_limit,schedule,best_strategy,calldata_bytes,bal_bytes,tx_access_list_bytes,total_payload_bytes,total_payload_mib,gas_used
0,60000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,937150,685460,0,1622610,1.547441,60000000
1,100000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,1562171,1142580,0,2704751,2.579452,99999944
2,150000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,2343421,1714004,0,4057425,3.869462,149999944
3,200000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,3124650,2285460,0,5410110,5.159483,200000000
4,300000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,4687171,3428308,0,8115479,7.739524,299999944
5,450000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,7030921,5142580,0,12173501,11.609555,449999944
6,60000000,glamsterdam_plus_8279,all_calldata_nonzero,937171,0,0,937171,0.893756,59999944
7,100000000,glamsterdam_plus_8279,all_calldata_nonzero,1562171,0,0,1562171,1.489802,99999944
8,150000000,glamsterdam_plus_8279,all_calldata_nonzero,2343421,0,0,2343421,2.234860,149999944
9,200000000,glamsterdam_plus_8279,all_calldata_nonzero,3124671,0,0,3124671,2.979918,199999944


,execution_gas_limit,schedule,strategy,total_payload_bytes,gas_used,is_best
0,60000000,glamsterdam_no_8279,all_calldata_nonzero,937171,59999944,False
1,60000000,glamsterdam_no_8279,sload_bal_only,913940,59999600,False
2,60000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,1622610,60000000,True
3,60000000,glamsterdam_no_8279,tx_access_list_plus_calldata,937171,59999944,False
4,100000000,glamsterdam_no_8279,all_calldata_nonzero,1562171,99999944,False
5,100000000,glamsterdam_no_8279,sload_bal_only,1523444,99998300,False
6,100000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,2704751,99999944,True
7,100000000,glamsterdam_no_8279,tx_access_list_plus_calldata,1562171,99999944,False
8,150000000,glamsterdam_no_8279,all_calldata_nonzero,2343421,149999944,False
9,150000000,glamsterdam_no_8279,sload_bal_only,2285364,149999300,False


## Part 2: Propagation Safety

In [3]:
best_with_propagation = add_propagation_times(
    best_table,
    fits=[EMPIRICAL_P90, CONSERVATIVE_P90],
    payload_col="total_payload_bytes",
)
display(best_with_propagation[best_display_cols + ["empirical_p90_ms", "conservative_p90_ms"]])

safe_caps = safe_payload_cap_sweep(
    windows_ms=[3000, 4000, 6000],
    safety_factors=[0.75, 1.0],
    fit=CONSERVATIVE_P90,
)

safe_caps_empirical = safe_payload_cap_sweep(
    windows_ms=[3000, 4000, 6000],
    safety_factors=[0.75, 1.0],
    fit=EMPIRICAL_P90,
)

display(safe_caps)
display(safe_caps_empirical)

,execution_gas_limit,schedule,best_strategy,calldata_bytes,bal_bytes,tx_access_list_bytes,total_payload_bytes,total_payload_mib,gas_used,empirical_p90_ms,conservative_p90_ms
0,60000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,937150,685460,0,1622610,1.547441,60000000,1270.968975,2036.239463
1,100000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,1562171,1142580,0,2704751,2.579452,99999944,1739.121771,3157.481261
2,150000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,2343421,1714004,0,4057425,3.869462,149999944,2324.311792,4559.031177
3,200000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,3124650,2285460,0,5410110,5.159483,200000000,2909.506572,5960.592490
4,300000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,4687171,3428308,0,8115479,7.739524,299999944,4079.895700,8763.714081
5,450000000,glamsterdam_no_8279,mixed_calldata_plus_cold_sloads,7030921,5142580,0,12173501,11.609555,449999944,5835.465765,12968.363829
6,60000000,glamsterdam_plus_8279,all_calldata_nonzero,937171,0,0,937171,0.893756,59999944,974.436282,1326.033624
7,100000000,glamsterdam_plus_8279,all_calldata_nonzero,1562171,0,0,1562171,1.489802,99999944,1244.822024,1973.616632
8,150000000,glamsterdam_plus_8279,all_calldata_nonzero,2343421,0,0,2343421,2.234860,149999944,1582.804202,2783.095392
9,200000000,glamsterdam_plus_8279,all_calldata_nonzero,3124671,0,0,3124671,2.979918,199999944,1920.786380,3592.574151


,fit,window_ms,safety_factor,safe_bandwidth_bytes
0,conservative_p90,3000,0.75,1828916
1,conservative_p90,3000,1.00,2552761
2,conservative_p90,4000,0.75,2552761
3,conservative_p90,4000,1.00,3517888
4,conservative_p90,6000,0.75,4000452
5,conservative_p90,6000,1.00,5448143


,fit,window_ms,safety_factor,safe_bandwidth_bytes
0,empirical_p90,3000,0.75,3885652
1,empirical_p90,3000,1.00,5619286
2,empirical_p90,4000,0.75,5619286
3,empirical_p90,4000,1.00,7930799
4,empirical_p90,6000,0.75,9086555
5,empirical_p90,6000,1.00,12553823


## Part 3: Candidate EIP-7999 Bandwidth Gas Limits

In [5]:
limit_candidates = eip7999_limit_candidates(
    safe_caps,
    gas_per_safe_byte=16,
)

limit_candidates_2 = eip7999_limit_candidates(
    safe_caps_empirical,
    gas_per_safe_byte=16,
)
display(limit_candidates)
display(limit_candidates_2)

,fit,window_ms,safety_factor,safe_bandwidth_bytes,bandwidth_gas_limit
0,conservative_p90,3000,0.75,1828916,29262656
1,conservative_p90,3000,1.00,2552761,40844176
2,conservative_p90,4000,0.75,2552761,40844176
3,conservative_p90,4000,1.00,3517888,56286208
4,conservative_p90,6000,0.75,4000452,64007232
5,conservative_p90,6000,1.00,5448143,87170288


,fit,window_ms,safety_factor,safe_bandwidth_bytes,bandwidth_gas_limit
0,empirical_p90,3000,0.75,3885652,62170432
1,empirical_p90,3000,1.00,5619286,89908576
2,empirical_p90,4000,0.75,5619286,89908576
3,empirical_p90,4000,1.00,7930799,126892784
4,empirical_p90,6000,0.75,9086555,145384880
5,empirical_p90,6000,1.00,12553823,200861168


## Part 4: Historical Replay Compatibility

In [ ]:
historical_path = PROJECT_ROOT / "data" / "xatu_calldata_50_blocks_22886891_22886940.csv"

if historical_path.exists():
    historical = pd.read_csv(historical_path)
    default_cap = int(
        safe_caps[
            (safe_caps["window_ms"] == 4000)
            & (safe_caps["safety_factor"] == 0.75)
        ]["safe_bandwidth_bytes"].iloc[0]
    )
    metering_config = BandwidthMeteringConfig(
        safe_bandwidth_bytes=default_cap,
        gas_per_safe_byte=16,
        bal_gas_per_byte=16,
    )
    historical_usage = historical_bandwidth_usage(historical, metering_config)
    display(historical_usage.head())
    display(historical_usage.describe())
else:
    print(f"No historical CSV found at {historical_path}")